# 🏦 BofA Institutional Flow EDA & Oracle Signal Analysis
This interactive Databricks-style notebook runs local DuckDB queries over the Medallion data tables (`silver_daily_broker_summary`, `gold_bofa_flow_metrics`) to inspect Bank of America's accumulation and distribution patterns on BIST.

In [ ]:
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Locate local DuckDB database
db_path = Path("../data/database/mdk_oracle.duckdb")
conn = duckdb.connect(str(db_path))
print(f"Connected to DuckDB: {db_path.resolve()}")

## 1. Inspect Available Database Tables

In [ ]:
tables_df = conn.execute("SHOW TABLES;").pl()
tables_df

## 2. Broker Turnover & Market Dominance Overview

In [ ]:
broker_summary_df = conn.execute("""
    SELECT 
        b.broker_name,
        s.broker_id,
        SUM(s.total_buy_tl) AS total_buy_tl,
        SUM(s.total_sell_tl) AS total_sell_tl,
        SUM(s.net_tl) AS net_flow_tl,
        SUM(s.total_buy_tl + s.total_sell_tl) AS total_turnover_tl
    FROM silver_daily_broker_summary s
    LEFT JOIN silver_brokers b ON s.broker_id = b.broker_id
    GROUP BY b.broker_name, s.broker_id
    ORDER BY total_turnover_tl DESC;
""").pl()

broker_summary_df

## 3. Gold Layer: BofA Flow vs Price Trend for AKBNK

In [ ]:
akbnk_gold = conn.execute("""
    SELECT 
        date_val,
        symbol,
        close_price,
        bofa_net_tl,
        bofa_cum_net_tl_20d,
        bofa_flow_zscore_20d,
        bofa_volume_share
    FROM gold_bofa_flow_metrics
    WHERE symbol = 'AKBNK'
    ORDER BY date_val ASC;
""").pl()

akbnk_gold.tail(10)

In [ ]:
fig = go.Figure()

# Price trace
fig.add_trace(go.Scatter(
    x=akbnk_gold['date_val'].to_list(),
    y=akbnk_gold['close_price'].to_list(),
    name="AKBNK Close Price (TL)",
    line=dict(color="#00d2d3", width=2),
    yaxis="y1"
))

# BofA Cumulative Net Flow
fig.add_trace(go.Bar(
    x=akbnk_gold['date_val'].to_list(),
    y=akbnk_gold['bofa_net_tl'].to_list(),
    name="BofA Daily Net TL",
    marker=dict(color=["#10ac84" if v > 0 else "#ee5253" for v in akbnk_gold['bofa_net_tl']]),
    yaxis="y2",
    opacity=0.6
))

fig.update_layout(
    title="AKBNK - Price vs BofA Institutional Net Flow",
    xaxis=dict(title="Date"),
    yaxis=dict(title="Price (TL)", side="left"),
    yaxis2=dict(title="BofA Net TL", side="right", overlaying="y", showgrid=False),
    template="plotly_dark",
    height=500
)
fig.show()

## 4. Latest Oracle Decision Signals

In [ ]:
signals_df = conn.execute("""
    SELECT 
        date_val, symbol, signal, confidence, bofa_net_tl, bofa_flow_zscore, summary
    FROM oracle_decision_signals
    ORDER BY confidence DESC;
""").pl()

signals_df